## Notebook 6 — AgentCore Platform Deep Dive: Gateway, Memory, Skills, Code Interpreter, Policy, Evaluations

> **Time budget:** ~45-60 minutes. This is the workshop's capstone notebook.

Notebook 4 built the `restaurant_concierge` harness end to end (prompt → memory → inline tool
→ Gateway tool). If you also did notebook 5 (optional), it added a `consult_events_specialist`
tool to that same harness — either way, this notebook continues on it, going one level deeper
into the AgentCore platform.

AgentCore is nine-plus components, not just Runtime/Gateway/Memory. This notebook covers, in
order:

1. **Gateway** — architecture, and a from-scratch walkthrough of *why* Gateway search matters
   (hundreds of tools, semantic search, 3x latency win) using raw MCP calls so you see exactly
   what the harness gets you for free.
2. **Memory** — a deeper look at the four strategy types (SEMANTIC, USER_PREFERENCE,
   SUMMARIZATION, EPISODIC), wired as harness config.
3. **Skills** — attaching `booking-ops` from S3, and testing skill changes without a redeploy.
4. **Code Interpreter** — the platform-native way to let an agent run arbitrary code.
5. **Policy** — a plain-English guardrail rule enforced before a tool call executes.
6. **Evaluations** — running a handful of test prompts through AgentCore's built-in evaluators.
7. **Observability** — traces and logs for everything above.
8. One-sentence mentions of Identity, Browser, and the emerging Optimization/Payments/Registry
   areas that are out of scope for this workshop.

Everything here is harness configuration: a model, tools, memory, and skills. There's no
orchestration loop to write and no agent framework required.

## Prerequisites

* Completion of notebook 4 (a deployed `restaurant_concierge` harness) — **required**
* Notebook 5 (agent-as-tool multi-agent) — **optional**; this notebook doesn't depend on
  anything it adds
* `bedrock-agentcore` / `bedrock-agentcore-starter-toolkit` (already in `requirements.txt`)
* AWS credentials with permission to manage AgentCore, Gateway, Lambda, IAM, Cognito

In [ ]:
import sys
sys.path.insert(0, "src")

from dotenv import load_dotenv
import os
import json
import time
import boto3
import requests
import utils

load_dotenv(".env")

aws_region = "us-east-1"
os.environ["AWS_REGION"] = aws_region
os.environ["AWS_DEFAULT_REGION"] = aws_region

session = boto3.Session()
user_session_name = "changeme"  # do not use special chars

agentcore_client = boto3.client("bedrock-agentcore-control", region_name=aws_region)
print(aws_region, user_session_name)

---
# Part 1 — AgentCore Gateway deep dive

| Information         | Details                                                              |
|:---------------------|:----------------------------------------------------------------------|
| Tutorial type        | Conversational                                                        |
| AgentCore services   | AgentCore Gateway, AgentCore Identity                                 |
| LLM model            | Anthropic Claude Sonnet 4.6 (via the `restaurant_concierge` harness)  |
| Tutorial components  | Creating and using a Lambda-backed AgentCore Gateway, raw MCP calls   |
| SDK used             | boto3 + raw MCP JSON-RPC over HTTP (framework-agnostic — no Strands)  |

Amazon Bedrock AgentCore Gateway provides unified connectivity between agents and the tools and
resources they need to interact with. Gateway plays multiple roles:

1. **Security Guard** — manages OAuth authorization so only valid users/agents access tools.
2. **Translator** — translates MCP requests into API requests and Lambda invocations.
3. **Composer** — combines multiple APIs, functions, and tools into a single MCP endpoint.
4. **Keychain** — injects the right credentials for the right tool.
5. **Researcher** — lets agents search across all their tools via natural language, so an agent
   can use thousands of tools without paying the prompt-token cost of listing them all.
6. **Infrastructure Manager** — fully serverless, built-in observability, no infra to run.

![How does it work](assets/images/gw-arch-overview.png)

**Note on authorizer type:** notebooks 4/5 used `--authorizer-type AWS_IAM` for simplicity (no
identity provider to stand up). This deep-dive notebook instead uses **Amazon Cognito** as a
Custom JWT authorizer, to show Gateway's inbound OAuth security model directly with a bearer
token you can inspect. Both are valid; AWS_IAM is the simpler default for most workshops.

## Why Gateway search matters

In a typical enterprise setting, agent builders encounter MCP servers with hundreds or even
thousands of tools. This poses real challenges: poor tool selection accuracy, higher cost, and
higher latency from excessive tool metadata in every prompt.

AgentCore Gateway provides a **built-in semantic search across tools**, which improves latency,
cost, and accuracy — you can see up to 3x better latency by keeping an agent focused on relevant
tools instead of the full set.

![How does it work](assets/images/gateway_tool_search.png)

This section builds a Gateway with several tool targets (some real, some synthetic — to simulate
a large tool catalog) and calls it directly with raw MCP JSON-RPC, so you can see exactly what's
happening before notebook 4/5's harness abstracts it away.

#### Helper functions for the control-plane APIs

In [ ]:
def read_apispec(json_file_path):
    try:
        with open(json_file_path, "r") as file:
            return json.load(file)
    except FileNotFoundError:
        return f"Error: File {json_file_path} not found"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"


def list_gateways():
    response = agentcore_client.list_gateways()
    print(json.dumps(response, indent=2, default=str))
    return response

#### Create Gateway helper function
Uses Amazon Cognito as its IdP, pulling the allowed client ID and discovery URL from the Cognito
user pool created below. Enables semantic search on the resulting Gateway.

In [ ]:
def create_gateway(gateway_name, gateway_desc):
    auth_config = {
        "customJWTAuthorizer": {
            "allowedClients": [cognito_response["client_id"]],
            "discoveryUrl": cognito_response["discovery_url"],
        }
    }
    search_config = {
        "mcp": {"searchType": "SEMANTIC", "supportedVersions": ["2025-03-26"]}
    }
    response = agentcore_client.create_gateway(
        name=gateway_name,
        roleArn=gateway_role_arn,
        authorizerType="CUSTOM_JWT",
        description=gateway_desc,
        protocolType="MCP",
        authorizerConfiguration=auth_config,
        protocolConfiguration=search_config,
    )
    return response["gatewayId"]


def create_gatewaytarget(gateway_id, target_name, target_descr, lambda_arn, api_spec):
    response = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=target_name,
        description=target_descr,
        targetConfiguration={
            "mcp": {"lambda": {"lambdaArn": lambda_arn, "toolSchema": {"inlinePayload": api_spec}}}
        },
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    return response["targetId"]

Let's look at the tool schemas for the two Lambda-backed targets we'll add: the restaurant booking API (from notebook 4/5's domain) and a small calculator API.

In [ ]:
with open("lambdas/restaurant/restaurant-api.json") as f:
    print(json.dumps(json.load(f), indent=4))

In [ ]:
with open("lambdas/calc/calc-api.json") as f:
    print(json.dumps(json.load(f)[0:3], indent=4))

In [ ]:
from IPython.display import display, Code

with open("lambdas/calc/lambda_function_code.py", "r") as f:
    display(Code(f.read(), language="python"))

In [ ]:
with open("lambdas/restaurant/lambda_function_code.py", "r") as f:
    display(Code(f.read(), language="python"))

#### Deploy the two Lambda functions that will back our Gateway targets

In [ ]:
calc_lambda_resp = utils.create_gateway_lambda(
    "lambdas/calc/lambda_function_code.zip", lambda_function_name=f"calc_lambda_gateway_{user_session_name}"
)
print("Calc Lambda ARN:", calc_lambda_resp.get("lambda_function_arn") if calc_lambda_resp else None)

In [ ]:
restaurant_lambda_resp = utils.create_gateway_lambda(
    "lambdas/restaurant/lambda_function_code.zip",
    lambda_function_name=f"restaurant_lambda_gateway_{user_session_name}",
)
print("Restaurant Lambda ARN:", restaurant_lambda_resp.get("lambda_function_arn") if restaurant_lambda_resp else None)

#### Set up Cognito (inbound OAuth for the Gateway) and an execution role

In [ ]:
cognito_response = utils.setup_cognito_user_pool()

In [ ]:
bearer_token = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)

In [ ]:
GATEWAY_AGENTCORE_ROLE_NAME = f"GatewaySearchAgentCoreRole{user_session_name}"
GATEWAY_AGENTCORE_POLICY_NAME = f"BedrockAgentPolicy{user_session_name}"

gateway_role_arn = utils.create_gateway_iam_role(
    lambda_arns=[calc_lambda_resp["lambda_function_arn"], restaurant_lambda_resp["lambda_function_arn"]],
    role_name=GATEWAY_AGENTCORE_ROLE_NAME,
    policy_name=GATEWAY_AGENTCORE_POLICY_NAME,
)

### Creating the Gateway and adding targets

We'll add a target for the restaurant Lambda and one for the calc Lambda, then pad the catalog
with a few synthetic extra calc targets so semantic search has hundreds of tools to search over
— exactly the scenario Gateway search is built for.

![How does it work](assets/images/gateway_secure_access.png)

In [ ]:
print(f"Create gateway with name: restaurant-concierge-search-{user_session_name}")
gatewayId = create_gateway(
    gateway_name=f"restaurant-concierge-search-{user_session_name}",
    gateway_desc="AgentCore Gateway deep-dive tutorial",
)
print(f"Gateway created with id: {gatewayId}.")

In [ ]:
restaurant_api_spec = read_apispec("lambdas/restaurant/restaurant-api.json")
restaurant_lambda_arn = restaurant_lambda_resp["lambda_function_arn"]

restaurantTargetId = create_gatewaytarget(
    gateway_id=gatewayId,
    lambda_arn=restaurant_lambda_arn,
    target_name="FoodTools",
    target_descr="Restaurant Tools",
    api_spec=restaurant_api_spec,
)
print(f"RestaurantTarget created with id: {restaurantTargetId}")

In [ ]:
calc_api_spec = read_apispec("lambdas/calc/calc-api.json")
calc_lambda_arn = calc_lambda_resp["lambda_function_arn"]

time.sleep(5)
calcTargetId = create_gatewaytarget(
    gateway_id=gatewayId,
    lambda_arn=calc_lambda_arn,
    target_name="CalcTools",
    target_descr="Calculation Tools",
    api_spec=calc_api_spec,
)
print(f"CalcTools Target created with id: {calcTargetId}")

To demonstrate the power of Gateway search, add a few more copies of the calc tools to simulate a much larger catalog:

In [ ]:
def add_more_tools(gatewayId):
    for label in ["Calc2", "Calc3", "Calc4"]:
        time.sleep(10)
        tid = create_gatewaytarget(
            gateway_id=gatewayId,
            lambda_arn=calc_lambda_arn,
            target_name=label,
            target_descr=f"{label} Tools",
            api_spec=calc_api_spec,
        )
        print(f"{label} Target created with id: {tid} on gateway: {gatewayId}")

add_more_tools(gatewayId=gatewayId)

In [ ]:
resp = agentcore_client.list_gateway_targets(gatewayIdentifier=gatewayId)
print(f"Gateway now has {len(resp['items'])} targets")

### Searching for tools from a Gateway

With the endpoint URL and JWT bearer token, we can call any MCP tool with raw JSON-RPC — including
the built-in `x_amz_bedrock_agentcore_search` tool.

In [ ]:
def get_gateway_endpoint(gateway_id):
    response = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    return response["gatewayUrl"]

gatewayEndpoint = get_gateway_endpoint(gateway_id=gatewayId)
print(gatewayEndpoint)

#### Using MCP Inspector against your Gateway

MCP Inspector is an open-source tool that can connect to any MCP server, list its tools, and
invoke them interactively. From your terminal, run `npx @modelcontextprotocol/inspector`, then
paste in the Gateway endpoint URL and JWT token above.

![MCP Inspector](assets/images/mcp_inspector.png)

In [ ]:
def invoke_gateway_tool(gateway_endpoint, jwt_token, tool_params):
    requestBody = {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": tool_params}
    response = requests.post(
        gateway_endpoint,
        json=requestBody,
        headers={"Authorization": f"Bearer {jwt_token}", "Content-Type": "application/json"},
    )
    return response.json()


def tool_search(gateway_endpoint, jwt_token, query):
    toolParams = {"name": "x_amz_bedrock_agentcore_search", "arguments": {"query": query}}
    toolResp = invoke_gateway_tool(gateway_endpoint, jwt_token, toolParams)
    return toolResp["result"]["structuredContent"]["tools"]

start = time.time()
results = tool_search(gatewayEndpoint, bearer_token, "divide two numbers")
print(f"Search took {time.time() - start:.2f}s, found {len(results)} tools")
for t in results:
    print("-", t["name"])

### What the harness gets you for free

Everything above — listing tools, calling `tools/call`, calling the built-in search tool — is
**raw MCP over HTTP**, useful for understanding what Gateway actually does. But notice: none of
it required Strands, a custom `MCPClient`, or any agent framework at all.

When you instead add this same Gateway as a harness tool (exactly like notebook 4 did:
`agentcore add tool --harness restaurant_concierge --type agentcore_gateway --gateway <name>`),
the harness's managed loop automatically:

- Lists the Gateway's tools and gives the model their schemas.
- Calls Gateway's semantic search first when the tool catalog is large, so the model only ever
  sees the handful of tools relevant to the current turn — the same 3x latency win demonstrated
  above, with zero client-side code.
- Handles the JWT/IAM auth handshake for you.

There is no more code to write here — this is the entire point of the harness-first approach:
what took ~30 lines of MCP client code above is a single `agentcore add tool` call once a Gateway
exists.

---
# Part 2 — Memory strategies, in depth

Notebook 4 wired all four memory strategies onto the `restaurant_concierge` harness and showed
one recall scenario. This section compares what each strategy is actually for, and re-runs a
fuller version of the recall demo.

| Strategy | What it captures | Good fit here |
|---|---|---|
| **SEMANTIC** | Standalone facts extracted from conversation ("guest is vegetarian") | Dietary notes, seating preferences |
| **USER_PREFERENCE** | Explicit stated preferences, weighted toward recency | "Always seat me by the window" |
| **SUMMARIZATION** | Rolling summary of long conversations | A guest who chats at length about an event booking |
| **EPISODIC** | Full turn-by-turn episodes, retrievable later | "What did I ask about last time I called?" |

All four are namespaced under `/users/{actorId}/...` and retrieved automatically by the harness
before the model sees the next prompt — this notebook does not write any retrieval code.

### Confirm memory is wired into the harness

This assumes notebook 4 already ran `agentcore add memory --name restaurant_conciergeMemory
--strategies SEMANTIC,USER_PREFERENCE,SUMMARIZATION,EPISODIC` and redeployed. If you're running
this notebook standalone, do that first (see notebook 4, Step 2).

In [ ]:
%%bash
cd RestaurantConciergeDemo
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore status --type harness

### Full recall demo: store in one session, retrieve from a brand-new one

In [ ]:
%%bash
cd RestaurantConciergeDemo

ACTOR_ID="guest-$(date +%s)"
echo "ACTOR_ID=$ACTOR_ID"
SESSION_A=$(uuidgen)   # AgentCore Harness requires session IDs >= 33 characters -- uuidgen, not a short literal

echo "---- Turn 1: store preference in SESSION_A ----"
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$SESSION_A" --actor-id "$ACTOR_ID" --stream \
  "Remember that I'm vegetarian and prefer a window seat. Reply with just: OK."

SESSION_B=$(uuidgen)
echo "---- Turn 2: brand-new session, same actor ----"
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$SESSION_B" --actor-id "$ACTOR_ID" --stream \
  "What are my dining preferences? Answer in one sentence using stored memory."

echo "ACTOR_ID for the next cell: $ACTOR_ID" 

Memory is keyed on `--actor-id`, **not** `--session-id`. `SESSION_A` and `SESSION_B` are
completely different conversations, but the platform retrieves the same stored preferences for
the same guest and injects them before the model sees the prompt — no custom DynamoDB table, no
retrieval code.

Inspect the raw stored memory record directly (replace `MEMORY_ID`/`ACTOR_ID` with values from your own run):

In [ ]:
%%bash
MEMORY_ID=$(python3 -c "
import json
with open('RestaurantConciergeDemo/agentcore/.cli/deployed-state.json') as f:
    d = json.load(f)
m = d['targets']['default']['resources']['memories']
print(list(m.values())[0]['memoryId'], end='')
")
echo "MEMORY_ID=$MEMORY_ID"

aws bedrock-agentcore retrieve-memory-records \
  --memory-id "$MEMORY_ID" \
  --region us-east-1 \
  --namespace "/users/REPLACE_WITH_ACTOR_ID/preferences" \
  --search-criteria '{"searchQuery":"vegetarian window seat"}' \
  --max-results 10

---
# Part 3 — Skills

A **Skill** is a Markdown file that adds domain behavior and guardrails without touching
orchestration code. Notebook 4 already wrote `config/skills/booking-ops/SKILL.md`. This section
attaches it two ways.

In [ ]:
with open("config/skills/booking-ops/SKILL.md") as f:
    print(f.read())

### Option A — S3 + AWS Console (persistent, production path)

As of this writing, the AgentCore CLI does **not** support attaching an S3-hosted skill directly
(this is a known, documented limitation — check `agentcore add skill --help` for the current
state before assuming this has changed). Upload the skill, then attach it through the console:

In [ ]:
%%bash
aws s3 cp config/skills/booking-ops/SKILL.md \
  s3://agentcore-workshop-restaurant-skills/skills/booking-ops/SKILL.md \
  --region us-east-1
aws s3 ls s3://agentcore-workshop-restaurant-skills/skills/ --recursive --region us-east-1

Then, in the AWS Console: **Amazon Bedrock → AgentCore → Harnesses → `restaurant_concierge` →
Edit → Skills → Add skill → S3 location** → enter the S3 URI above → **Save**, and wait ~2-3
minutes for the harness status to return to `READY`.

**Important:** all skills must be added in a single edit session — the "Add skill" button
disappears after the first skill is added; to add more later, edit the existing configuration.

### Option B — local override for quick iteration

For testing a skill you're still editing, pass it directly at invoke time — no redeploy needed:

In [ ]:
%%bash
cd RestaurantConciergeDemo

echo "---- Before the skill is active ----"
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$(uuidgen)" --stream \
  "I'd like a table for 6 tomorrow at 20:00 under the name Sara, is that possible?"

echo "---- With the skill applied as a local override ----"
mkdir -p app/restaurant_concierge/skills/booking-ops
cp ../config/skills/booking-ops/SKILL.md app/restaurant_concierge/skills/booking-ops/SKILL.md

AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$(uuidgen)" \
  --skills app/restaurant_concierge/skills/booking-ops \
  --stream \
  "I'd like a table for 6 tomorrow at 20:00 under the name Sara, is that possible?" 

Compare the two responses: the skill should reject the relative date ("tomorrow") and ask for an explicit `YYYY-MM-DD` date, and flag the party of 6 as worth mentioning to a human host per the skill's guidance.

---
# Part 4 — Code Interpreter

Notebooks 4-6 (this one, before this rewrite) used a hand-built `calc` Lambda behind a Gateway
target for arithmetic. AgentCore's **Code Interpreter** is the platform-native replacement: a
managed, sandboxed code-execution tool the model can call directly — no Lambda to write, deploy,
or maintain.

> **Flagging an assumption:** Code Interpreter is GA, but the exact harness `tools[]` entry shape
> for wiring it in is newer surface than the Gateway/Memory patterns above. Before running this in
> a real workshop, confirm the tool `type` value and config keys with
> `agentcore add tool --type code_interpreter --help` (or the current AgentCore harness JSON
> schema docs) — the sketch below is the best-known shape at time of writing, not a verified copy
> of prior working output like the Gateway/Memory sections.

In [ ]:
%%bash
cd RestaurantConciergeDemo
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore add tool \
  --harness restaurant_concierge \
  --type code_interpreter \
  --name kitchen_math

AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore deploy

In [ ]:
%%bash
cd RestaurantConciergeDemo
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$(uuidgen)" --stream \
  "Calculate the total bill for a table of 6 ordering three of today's specials at \$28 each, plus a 20% service charge. Show your work." 

Compare to the `calc` Lambda + Gateway target from Part 1 of this notebook: that required writing and deploying a function for four hardcoded operations. Code Interpreter runs arbitrary Python in a managed sandbox — no Lambda, no fixed operation list, and it scales to any calculation the model needs to reason about precisely instead of estimating.

---
# Part 5 — Policy

AgentCore Policy (GA 2026-03-03) lets you author guardrails in plain English, compiled to Cedar
policies under the hood, enforced by the platform before a tool call executes — independent of
whatever the model or the SKILL.md says.

> **Flagging an assumption:** this is one of the newest AgentCore surfaces referenced in
> `UPGRADE_PLAN.md`. The exact CLI/config shape below is a best-effort sketch based on the
> platform's stated natural-language-authoring capability — verify against
> `agentcore add policy --help` and the current AgentCore Policy docs before relying on it live.
> If the CLI surface differs, the concept to demonstrate is unchanged: a plain-English rule that
> blocks a `request_booking` call for large parties without human approval.

In [ ]:
%%bash
cd RestaurantConciergeDemo
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore add policy \
  --harness restaurant_concierge \
  --name large-party-guardrail \
  --rule "Never confirm or execute a request_booking call for more than 12 guests without explicit human approval."

AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore deploy

In [ ]:
%%bash
cd RestaurantConciergeDemo

echo "---- Should be blocked (party of 20) ----"
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$(uuidgen)" --stream \
  "Book a table for 20 people tonight at 20:00 under the name Marco."

echo "---- Should be allowed (party of 4) ----"
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore invoke \
  --session-id "$(uuidgen)" --stream \
  "Book a table for 4 people tonight at 20:00 under the name Marco." 

This is a *hard* guardrail enforced by the platform, layered on top of the SKILL.md's *soft* guidance from Part 3 (which already asked the model to escalate parties of 6+) — Policy is what you reach for when "the model usually follows the skill" isn't a strong enough guarantee.

---
# Part 6 — Evaluations

"You built it, now prove it works." AgentCore Evaluations (GA 2026-03-31) ships 13 built-in
evaluators (tool-selection accuracy, helpfulness, groundedness, etc.) that can run against a set
of prepared prompts.

> **Flagging an assumption:** as with Policy, this is newer surface — confirm the exact
> `agentcore evaluate`/`agentcore add evaluation` syntax against current docs before a live
> workshop run. The shape below is a best-effort sketch, not verified working output.

In [ ]:
%%bash
cat > RestaurantConciergeDemo/eval-prompts.jsonl << 'EOF'
{"prompt": "What are today's specials?", "expected_tool": "restaurant_gateway_data"}
{"prompt": "Book a table for 4 tonight at 20:00 under the name Elena.", "expected_tool": "request_booking"}
{"prompt": "Book a table for 30 people for a wedding next week.", "expected_behavior": "escalate_or_block"}
{"prompt": "What's 18% service charge on a $140 bill?", "expected_tool": "kitchen_math"}
{"prompt": "Do you replace a human host?", "expected_behavior": "no_tool_call"}
EOF

cd RestaurantConciergeDemo
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore evaluate \
  --harness restaurant_concierge \
  --prompts ../eval-prompts.jsonl \
  --evaluators tool_selection_accuracy,helpfulness

Read the evaluator output for tool-selection accuracy and helpfulness scores per prompt. This is the capstone check: did notebooks 4-6's configuration actually produce the behavior the SKILL.md and Policy rule intended?

---
# Part 7 — Observability

Every invocation across this entire workshop has been traced automatically: model calls, tool
calls, memory reads, all with timing and token counts.

In [ ]:
%%bash
cd RestaurantConciergeDemo
RUNTIME_ID=$(python3 -c "
import json
with open('agentcore/.cli/deployed-state.json') as f:
    d = json.load(f)
arn = d['targets']['default']['resources']['harnesses']['restaurant_concierge']['agentRuntimeArn']
print(arn.split('/')[-1], end='')
")
echo "Runtime ID: $RUNTIME_ID"

AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore logs --runtime "$RUNTIME_ID" --since 1h
AWS_REGION=us-east-1 AWS_DEFAULT_REGION=us-east-1 agentcore traces list --runtime "$RUNTIME_ID"

Cost controls (`maxIterations`, `timeoutSeconds`) live in `harness.json` and can be overridden per invocation with `--max-iterations`.

---
# Part 8 — Everything else, in one sentence each

- **Identity** (OBO token exchange, custom claims): only matters when your agent must call a
  third-party API *as* the signed-in user — the restaurant concierge never does that, so there's
  nothing to demo here.
- **Browser**: a managed, sandboxed way for an agent to interact with real websites — not needed
  for a restaurant-facts-and-bookings domain.
- **Optimization, Payments, Registry, A2A**: emerging AgentCore areas (PrivateLink networking,
  x402 microtransactions, org-wide agent catalogs, agent-to-agent protocol) worth watching as the
  platform matures, but out of scope for a single time-boxed workshop session.

---
# Cleanup

Removes the Gateway/Lambda/IAM/Cognito resources created in Part 1 of *this* notebook. It does
**not** tear down the `restaurant_concierge` harness/memory/gateway from notebooks 4-5 — see
those notebooks (or the original demo's `make clean`) for that.

In [ ]:
clients = {
    service: boto3.client(service, region_name=aws_region)
    for service in ["bedrock-agentcore-control", "lambda", "iam", "cognito-idp"]
}

def safe_delete(func, **kwargs):
    try:
        func(**kwargs)
        return True
    except Exception as e:
        if "NotFound" not in str(e) and "NoSuchEntity" not in str(e):
            print(f"Error during cleanup: {e}")
        return False


def delete_gatewaytarget(gateway_id):
    response = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)
    for target in response["items"]:
        print(f"Deleting target {target['name']} ({target['targetId']})")
        agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target["targetId"])
        time.sleep(20)


if "gatewayId" in locals():
    delete_gatewaytarget(gateway_id=gatewayId)
    safe_delete(agentcore_client.delete_gateway, gatewayIdentifier=gatewayId)

    for arn in [calc_lambda_resp["lambda_function_arn"], restaurant_lambda_resp["lambda_function_arn"]]:
        if utils.delete_gateway_lambda(arn):
            print(f"Deleted Lambda: {arn}")

    if utils.delete_gateway_iam_role():
        print("Gateway IAM role deleted")

print("Cleanup complete.")